In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/qm9-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "qm9_gap.py"

# 用結構估計 QM9 的 HOMO–LUMO gap
QM9 原始資料有 133,885 個分子，目標為 LUMO 能量減去 HOMO 能量（hartree）。原始論文 DOI 10.1038/sdata.2014.22。第一次需下載 51 MiB 的官方資料並產生指紋快取。

中間的程式格寫出一支暫存的 Python 程式，最後一格在本機書籍環境或 Colab 獨立環境執行它並顯示圖片。小練習可以直接加在程式底端，再執行這兩格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import logging
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem, rdBase
from rdkit.Chem import rdFingerprintGenerator
from scipy.sparse import csr_matrix
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

SEED = 42
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

class MorganRadius2Bits1024(dc.feat.Featurizer):
    def _featurize(self, mol, **kwargs):
        # SDF 有顯式氫原子。移除後與一般從 SMILES 建立的結構使用同一套慣例。
        return generator.GetFingerprintAsNumPy(Chem.RemoveHs(mol))

cache = Path(os.environ.get("QM9_CACHE", Path(tempfile.gettempdir()) / "cheminfo-qm9-cache"))
cache.mkdir(parents=True, exist_ok=True)

# DeepChem 會略過無法通過 RDKit 化學檢查的結構。保留真正的錯誤訊息，
# 但略過逐列警告，避免第一次載入時出現數千行重複輸出。
logging.getLogger("deepchem.utils.data_utils").setLevel(logging.ERROR)
with rdBase.BlockLogs():
    tasks, (train, valid, test), _ = dc.molnet.load_qm9(
        featurizer=MorganRadius2Bits1024(), splitter="scaffold", transformers=[],
        data_dir=str(cache), save_dir=str(cache), reload=True,
    )
gap = tasks.index("gap")
print("train, valid, test:", len(train), len(valid), len(test))
for split in (train, valid, test):
    labels = split.y
    columns = [tasks.index(name) for name in ("homo", "lumo", "gap")]
    assert np.all(split.w[:, columns] > 0)
    np.testing.assert_allclose(labels[:, gap],
                               labels[:, tasks.index("lumo")] - labels[:, tasks.index("homo")],
                               rtol=0, atol=1.1e-4)

X_train = csr_matrix(train.X)
y_train = train.y[:, gap]
# 預先固定模型與超參數，驗證集保留給練習。
model = Ridge(alpha=1.0, solver="lsqr", tol=1e-6)
model.fit(X_train, y_train)
y_test = test.y[:, gap]
y_pred = model.predict(csr_matrix(test.X))
mae = mean_absolute_error(y_test, y_pred)
median_train = np.median(y_train)
baseline = mean_absolute_error(y_test, np.full_like(y_test, median_train))
print(f"Test MAE (hartree): Ridge = {mae:.5f}; training median = {baseline:.5f}")

# 全部測試點都納入 MAE。畫圖也使用全組，隨機順序只影響點重疊時的前後位置。
order = np.random.default_rng(SEED).permutation(len(y_test))
fig, ax = plt.subplots(figsize=(6.4, 5.4), layout="constrained")
limits = [min(y_test.min(), y_pred.min()) - 0.01, max(y_test.max(), y_pred.max()) + 0.01]
ax.scatter(y_test[order], y_pred[order], s=6, alpha=0.16, color="#296b84", edgecolors="none")
ax.plot(limits, limits, color="#b74f26", lw=1.4, label="Equal values")
ax.set(xlim=limits, ylim=limits, aspect="equal",
       xlabel="QM9 orbital gap (hartree)", ylabel="Predicted orbital gap (hartree)")
ax.legend(loc="upper left", frameon=False)
ax.set_title("Morgan fingerprint + ridge regression", loc="left")
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / "05-08-qm9-gap.png", dpi=165)
plt.close(fig)

In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "05-08-qm9-gap.png")))